In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE  = 'analysis_dataset_2021_07.csv'
OUTPUT_FILE = 'ai_scenario_results_2021_07.csv'
PLOT_FILE   = 'ai_scenario_gradient_2021_07.png'

GRADIENT_STEPS = 200

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading analysis dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})

# Restrict to counties with complete data (same as OLS sample)
df_main = df[
    df['mortality_rate'].notna() &
    df['rucc_2023'].notna() &
    df['median_household_income'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

print(f"Analysis sample: {len(df_main):,} counties")

# ============================================================
# Step 1b: Fit M2 within this script (rather than hardcoding its
# beta as a literal copied from script 3's output). This keeps this
# script self-consistent with whatever INPUT_FILE is loaded above,
# instead of risking a stale beta if the input dataset changes but
# this value isn't manually updated (as happened once already).
# ============================================================
df_main['log_density_rad'] = np.log(df_main['density_radiologist'] + 1)
df_main['income_10k'] = df_main['median_household_income'] / 10000
df_main['rucc_2023'] = pd.to_numeric(df_main['rucc_2023'], errors='coerce')

m2_check = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
BETA = round(m2_check.params['log_density_rad'], 4)
print(f"M2 beta (recomputed within this script for consistency): {BETA}")

# ============================================================
# Step 2: Compute AI capacity scenarios and shortage cutoffs
# dynamically from the observed density distribution.
#
# Changed from hardcoded literals (1.2716 / 5.1688 / 10.9232, taken
# from the 2026 NPPES version's distribution) to computing these
# directly from df_main each time the script runs. This makes the
# scenario values, shortage cutoff, gradient range, and every label
# that references them self-updating for whatever dataset is loaded,
# instead of requiring 8 separate hardcoded spots to be kept in sync
# by hand.
# ============================================================
nonzero_density = df_main.loc[df_main['density_radiologist'] > 0, 'density_radiologist']

LOW_VAL      = round(nonzero_density.quantile(0.25), 4)
MODERATE_VAL = round(nonzero_density.quantile(0.50), 4)
HIGH_VAL     = round(nonzero_density.quantile(0.75), 4)

print(f"\nAI scenario values (from observed nonzero-county density distribution):")
print(f"  Nonzero counties: {len(nonzero_density):,}, max density: {nonzero_density.max():.1f}")
print(f"  Low (25th pct):      {LOW_VAL}")
print(f"  Moderate (median):   {MODERATE_VAL}")
print(f"  High (75th pct):     {HIGH_VAL}")

SCENARIOS = {
    'Baseline (ΔAI = 0)':              0.0,
    f'Low (ΔAI = {LOW_VAL:.1f})':      LOW_VAL,
    f'Moderate (ΔAI = {MODERATE_VAL:.1f})': MODERATE_VAL,
    f'High (ΔAI = {HIGH_VAL:.1f})':    HIGH_VAL,
}

# Shortage subgroup definitions
# Primary:   extreme shortage (density = 0)
# Robustness: severe shortage (density < 25th percentile of nonzero counties)
EXTREME_SHORTAGE_CUTOFF = 0.0
SEVERE_SHORTAGE_CUTOFF  = LOW_VAL

# Continuous gradient range
GRADIENT_MAX = HIGH_VAL

# Labels used throughout (built once here so every downstream dict/plot/print
# uses the same, currently-correct cutoff value in its text)
EXTREME_LABEL     = 'Extreme shortage (density = 0)'
SEVERE_LABEL      = f'Severe shortage (density < {SEVERE_SHORTAGE_CUTOFF:.2f})'
NONSHORTAGE_LABEL = f'Non-shortage (density >= {SEVERE_SHORTAGE_CUTOFF:.2f})'

n_extreme_ai = (df_main['density_radiologist']==0).sum()
print(f"Extreme shortage (density=0):  {n_extreme_ai:,} counties ({n_extreme_ai/len(df_main)*100:.1f}% of analytic sample)")
print(f"Severe shortage (density<{SEVERE_SHORTAGE_CUTOFF:.2f}): {(df_main['density_radiologist']<SEVERE_SHORTAGE_CUTOFF).sum():,} counties")

# ============================================================
# Step 3: Define AI capacity injection function
# ============================================================
def compute_mortality_change(density, delta_ai, beta=BETA):
    """
    Compute predicted mortality rate change under AI capacity injection.

    Mechanism: AI adds ΔAI radiologists per 100,000 population (additive).
    This allows zero-density counties to benefit from AI deployment,
    consistent with teleradiology and AI-enabled diagnostic services.

    density_new = density + ΔAI
    Δlog = log(density_new + 1) - log(density + 1)
    Δmortality = β × Δlog
    """
    log_density_original = np.log(density + 1)
    log_density_new      = np.log(density + delta_ai + 1)
    delta_log            = log_density_new - log_density_original
    delta_mortality      = beta * delta_log
    return delta_mortality

# ============================================================
# Step 4: Scenario analysis — all counties
# ============================================================
print("\n=== Scenario Analysis: All Counties ===")
results = []

for scenario_name, delta_ai in SCENARIOS.items():
    df_main[f'delta_mortality_{scenario_name}'] = compute_mortality_change(
        df_main['density_radiologist'], delta_ai
    )
    mean_change = df_main[f'delta_mortality_{scenario_name}'].mean()
    median_change = df_main[f'delta_mortality_{scenario_name}'].median()
    print(f"\n{scenario_name}")
    print(f"  Mean mortality change:   {mean_change:.4f} per 100,000")
    print(f"  Median mortality change: {median_change:.4f} per 100,000")

# ============================================================
# Step 5: Shortage subgroup analysis
# ============================================================
print("\n=== Shortage Subgroup Analysis ===")

subgroups = {
    EXTREME_LABEL:     df_main['density_radiologist'] == 0,
    SEVERE_LABEL:      df_main['density_radiologist'] < SEVERE_SHORTAGE_CUTOFF,
    NONSHORTAGE_LABEL: df_main['density_radiologist'] >= SEVERE_SHORTAGE_CUTOFF,
}

scenario_summary = []

for subgroup_name, mask in subgroups.items():
    subgroup = df_main[mask].copy()
    subgroup_pop = subgroup['population_2021'].sum()
    print(f"\n{subgroup_name} — N = {len(subgroup):,}")
    print(f"  Total population: {subgroup_pop/1e6:.1f} million")
    print(f"  Mean baseline mortality: {subgroup['mortality_rate'].mean():.2f}")

    for scenario_name, delta_ai in SCENARIOS.items():
        delta_mort = compute_mortality_change(
            subgroup['density_radiologist'], delta_ai
        )
        mean_change  = delta_mort.mean()
        total_deaths = mean_change * subgroup['population_2021'].sum() / 100_000
        print(f"  {scenario_name}: Δmortality = {mean_change:.4f} "
              f"(≈ {abs(total_deaths):,.0f} deaths averted per year)")

        scenario_summary.append({
            'subgroup':        subgroup_name,
            'scenario':        scenario_name,
            'delta_ai':        delta_ai,
            'n_counties':      len(subgroup),
            'population_total': int(subgroup_pop),
            'mean_baseline_mortality': subgroup['mortality_rate'].mean().round(4),
            'mean_delta_mortality':    round(mean_change, 4),
            'estimated_deaths_averted': round(abs(total_deaths), 1)
        })

results_df = pd.DataFrame(scenario_summary)

# ============================================================
# Step 6: Continuous gradient sensitivity analysis
# ============================================================
print("\n=== Continuous Gradient Sensitivity Analysis ===")
gradient_values = np.linspace(0, GRADIENT_MAX, GRADIENT_STEPS)

gradient_results = {}
for subgroup_name, mask in subgroups.items():
    subgroup = df_main[mask].copy()
    mean_changes = []
    for delta_ai in gradient_values:
        delta_mort = compute_mortality_change(
            subgroup['density_radiologist'], delta_ai
        )
        mean_changes.append(delta_mort.mean())
    gradient_results[subgroup_name] = mean_changes

print("Gradient analysis complete.")
print(f"Range: ΔAI = 0 to {GRADIENT_MAX:.1f}")
print(f"Steps: {GRADIENT_STEPS}")

# ============================================================
# Step 7: Plot continuous gradient (publication-ready version)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

# Plot lines: extreme=thick solid, severe=dashed, non-shortage=solid
plot_styles = {
    EXTREME_LABEL:     dict(color='#d62728', linewidth=2.5, linestyle='-'),
    SEVERE_LABEL:      dict(color='#ff7f0e', linewidth=1.5, linestyle='--'),
    NONSHORTAGE_LABEL: dict(color='#1f77b4', linewidth=2.0, linestyle='-'),
}

for subgroup_name, mean_changes in gradient_results.items():
    # Flip sign: negative change → positive reduction
    reductions = [-x for x in mean_changes]
    ax.plot(gradient_values, reductions,
            label=subgroup_name,
            **plot_styles[subgroup_name])

# Mark discrete scenario points (skip ΔAI=0)
scenario_line_colors = {
    f'Low (ΔAI = {LOW_VAL:.1f})':      'green',
    f'Moderate (ΔAI = {MODERATE_VAL:.1f})': 'purple',
    f'High (ΔAI = {HIGH_VAL:.1f})':    'red',
}
for scenario_name, delta_ai in SCENARIOS.items():
    if delta_ai == 0:
        continue
    color = scenario_line_colors[scenario_name]
    ax.axvline(x=delta_ai, color=color, linestyle='--', alpha=0.4, linewidth=1)
    ax.text(delta_ai + 0.1, 0.1, f'ΔAI={delta_ai:.2f}',
            fontsize=8, color=color, rotation=90, va='bottom')

# Annotation: highlight gap at the Moderate scenario value
# Fix: compute exactly at delta_ref = MODERATE_VAL using the same
# compute_mortality_change() function used for the official scenario
# results, rather than looking up the nearest of the 200 discrete
# gradient grid points (which could differ slightly from the exact
# value due to grid spacing, e.g. 3.91 vs the official 3.92).
delta_ref = MODERATE_VAL

shortage_val    = -compute_mortality_change(
    df_main.loc[subgroups[EXTREME_LABEL], 'density_radiologist'], delta_ref
).mean()
nonshortage_val = -compute_mortality_change(
    df_main.loc[subgroups[NONSHORTAGE_LABEL], 'density_radiologist'], delta_ref
).mean()

ax.annotate('', xy=(delta_ref, shortage_val), xytext=(delta_ref, nonshortage_val),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(delta_ref + 0.2, (shortage_val + nonshortage_val) / 2,
        f'Shortage: {shortage_val:.2f}\nNon-shortage: {nonshortage_val:.2f}',
        fontsize=8.5, color='black', va='center')

ax.set_xlabel('Hypothetical AI-Equivalent Capacity Increment (ΔAI, radiologist-equivalent units per 100,000 population)', fontsize=12)
ax.set_ylabel('Modeled Cancer Mortality Differential\n(per 100,000 population)', fontsize=12)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=300, bbox_inches='tight')
print(f"\nPlot saved to: {PLOT_FILE}")

# ============================================================
# Step 8: Save results
# ============================================================
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"Results saved to: {OUTPUT_FILE}")

print("\n" + "="*60)
print("KEY FINDINGS SUMMARY")
print("="*60)
extreme = results_df[results_df['subgroup'] == EXTREME_LABEL]
for _, row in extreme.iterrows():
    print(f"{row['scenario']}: "
          f"Δmortality = {row['mean_delta_mortality']:.4f}, "
          f"≈ {row['estimated_deaths_averted']:,.0f} deaths averted/year")

print("\nDONE. Proceed to Script 5 (spatial regression robustness check).")

Loading analysis dataset...
Analysis sample: 3,073 counties
M2 beta (recomputed within this script for consistency): -2.0054

AI scenario values (from observed nonzero-county density distribution):
  Nonzero counties: 1,739, max density: 392.0
  Low (25th pct):      2.418
  Moderate (median):   6.0451
  High (75th pct):     11.6341
Extreme shortage (density=0):  1,334 counties (43.4% of analytic sample)
Severe shortage (density<2.42): 1,769 counties

=== Scenario Analysis: All Counties ===

Baseline (ΔAI = 0)
  Mean mortality change:   0.0000 per 100,000
  Median mortality change: -0.0000 per 100,000

Low (ΔAI = 2.4)
  Mean mortality change:   -1.5470 per 100,000
  Median mortality change: -2.0758 per 100,000

Moderate (ΔAI = 6.0)
  Mean mortality change:   -2.5888 per 100,000
  Median mortality change: -3.4328 per 100,000

High (ΔAI = 11.6)
  Mean mortality change:   -3.5097 per 100,000
  Median mortality change: -4.5637 per 100,000

=== Shortage Subgroup Analysis ===

Extreme shortag